# Rural Road Extraction - MobileViT v2 Pipeline

This is a fully self-contained notebook designed to run on Kaggle T4 GPU accelerators. It downloads the dataset, handles any missing masks, dynamically splits the dataset for validation if needed, uses PyTorch Mixed Precision (AMP), and trains the MobileViT v2 network with clDice loss.

In [22]:
!pip install gdown albumentations opencv-python-headless -q

import gdown
import os

file_id = '1OI0XJ1-ejxd0JS45hBzJbAYe9_2hJJwN'
url = f'https://drive.google.com/uc?id={file_id}'
output = '/kaggle/working/archive.zip'

if not os.path.exists(output):
    print("Downloading dataset...")
    gdown.download(url, output, quiet=False)
else:
    print("Dataset already downloaded.")

if not os.path.exists('/kaggle/working/dataset'):
    print("Extracting dataset...")
    !unzip -q /kaggle/working/archive.zip -d /kaggle/working/dataset
    print("Extraction complete.")
else:
    print("Dataset already extracted.")

Dataset already downloaded.
Dataset already extracted.


In [23]:
import os
import sys
import math
import time
import json
import datetime
import random
from pathlib import Path
from typing import Dict, Tuple, List

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from tqdm import tqdm

import albumentations as A
from albumentations.pytorch import ToTensorV2

# Paths Configuration
TRAIN_IMG_DIR = '/kaggle/working/dataset/train'
TRAIN_MASK_DIR = '/kaggle/working/dataset/train'
VAL_IMG_DIR = '/kaggle/working/dataset/valid'
VAL_MASK_DIR = '/kaggle/working/dataset/valid'
OUTPUT_DIR = '/kaggle/working/model'

# Training Configuration
EPOCHS = 50
BATCH_SIZE = 16
LR = 1e-3
WIDTH_MULT = 1.0
NUM_WORKERS = min(4, os.cpu_count() or 2)
RUN_NAME = "kaggle-mobilevit-v2-run1"

In [24]:
def _wandb_available() -> bool:
    api_key = os.environ.get("WANDB_API_KEY", "").strip()
    if not api_key:
        return False
    try:
        import wandb
        return True
    except ImportError:
        return False

class WandbLogger:
    def __init__(
        self,
        project: str = "rural-road-extraction",
        run_name: str | None = None,
        config: dict | None = None,
        output_dir: str = "outputs",
        tags: list | None = None,
    ):
        self._use_wandb = _wandb_available()
        self._run_name = run_name or f"run_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"
        self._output_dir = Path(output_dir)
        self._output_dir.mkdir(parents=True, exist_ok=True)
        self._local_log_path = self._output_dir / "wandb_local.json"
        self._local_records: list[dict] = []
        self._step = 0

        if self._use_wandb:
            try:
                import wandb
                self._run = wandb.init(
                    project=project,
                    name=self._run_name,
                    config=config or {},
                    tags=tags or [],
                    reinit=True,
                )
                print(f"[WandbLogger] Connected to W&B project='{project}' run='{self._run_name}'")
            except Exception as e:
                print(f"[WandbLogger] W&B init failed ({e}), falling back to local logging.")
                self._use_wandb = False
                self._run = None
        else:
            self._run = None
            print(f"[WandbLogger] W&B not available. Logging locally to {self._local_log_path}")

    def log_metric(self, name: str, value: float, step: int | None = None) -> None:
        self.log_metrics({name: value}, step=step)

    def log_metrics(self, metrics: dict, step: int | None = None) -> None:
        if step is None:
            self._step += 1
            step = self._step

        record = {"step": step, "timestamp": time.time(), **metrics}

        if self._use_wandb:
            try:
                import wandb
                wandb.log(metrics, step=step)
            except Exception as e:
                print(f"[WandbLogger] Failed to log metrics to W&B: {e}")

        self._local_records.append(record)
        self._flush_local()

    def log_artifact(
        self,
        file_path: str,
        artifact_type: str = "model",
        name: str | None = None,
        metadata: dict | None = None,
    ) -> None:
        file_path = str(file_path)
        name = name or Path(file_path).stem

        if self._use_wandb:
            try:
                import wandb
                artifact = wandb.Artifact(name=name, type=artifact_type, metadata=metadata or {})
                artifact.add_file(file_path)
                self._run.log_artifact(artifact)
                print(f"[WandbLogger] Artifact logged: {file_path}")
            except Exception as e:
                print(f"[WandbLogger] Failed to log artifact ({e}). Path: {file_path}")
        else:
            print(f"[WandbLogger] Artifact (local only): {file_path}")
            self._local_records.append({
                "artifact": file_path,
                "type": artifact_type,
                "timestamp": time.time(),
            })
            self._flush_local()

    def log_config(self, config: dict) -> None:
        if self._use_wandb and self._run is not None:
            try:
                import wandb
                wandb.config.update(config)
            except Exception:
                pass
        self._local_records.append({"config": config, "timestamp": time.time()})
        self._flush_local()

    def log_summary(self, summary: dict) -> None:
        if self._use_wandb and self._run is not None:
            try:
                import wandb
                for k, v in summary.items():
                    wandb.run.summary[k] = v
            except Exception:
                pass
        self._local_records.append({"summary": summary, "timestamp": time.time()})
        self._flush_local()

    def finish(self) -> None:
        if self._use_wandb and self._run is not None:
            try:
                import wandb
                wandb.finish()
            except Exception:
                pass
        self._flush_local()
        print(f"[WandbLogger] Run finished. Local log: {self._local_log_path}")

    def _flush_local(self) -> None:
        try:
            with open(self._local_log_path, "w") as f:
                json.dump(self._local_records, f, indent=2, default=str)
        except Exception:
            pass

In [25]:
class DeepGlobeDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.transform = transform

        # Check for matching image-mask pairs to avoid NoneType errors
        self.ids = []
        if os.path.exists(image_dir) and os.path.exists(mask_dir):
            for f in os.listdir(image_dir):
                if f.endswith(".jpg"):
                    img_id = f.split("_")[0]
                    mask_name = f"{img_id}_mask.png"
                    if os.path.exists(os.path.join(mask_dir, mask_name)):
                        self.ids.append(img_id)

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, index):
        img_id = self.ids[index]
        img_path = os.path.join(self.image_dir, f"{img_id}_sat.jpg")
        mask_path = os.path.join(self.mask_dir, f"{img_id}_mask.png")

        image = cv2.imread(img_path)
        if image is None:
            image = np.zeros((256, 256, 3), dtype=np.uint8)
        else:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if os.path.exists(mask_path):
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        else:
            mask = None

        if mask is None:
            h, w = image.shape[:2]
            mask = np.zeros((h, w), dtype=np.uint8)

        if self.transform is not None:
            augmentations = self.transform(image=image, mask=mask)
            image = augmentations["image"]
            mask = augmentations["mask"]

        if torch.is_tensor(mask):
            mask = mask.to(dtype=torch.float32) / 255.0
        else:
            mask = torch.tensor(mask, dtype=torch.float32) / 255.0

        mask = mask.unsqueeze(0)
        return image, mask

def get_train_transforms():
    return A.Compose([
        A.RandomCrop(width=256, height=256),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomBrightnessContrast(p=0.2),
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.3),
        A.CoarseDropout(max_holes=8, max_height=48, max_width=48, min_holes=2, min_height=16, min_width=16, fill_value=0, mask_fill_value=None, p=0.4),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225], max_pixel_value=255.0),
        ToTensorV2(),
    ])

def get_val_transforms():
    return A.Compose([
        A.Resize(height=256, width=256),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225], max_pixel_value=255.0),
        ToTensorV2(),
    ])

In [26]:
def soft_erode(img: torch.Tensor) -> torch.Tensor:
    if img.dim() == 3:
        img = img.unsqueeze(1)
    p_v = -F.max_pool2d(-img, kernel_size=(3, 1), stride=1, padding=(1, 0))
    p_h = -F.max_pool2d(-img, kernel_size=(1, 3), stride=1, padding=(0, 1))
    return torch.min(p_v, p_h)

def soft_dilate(img: torch.Tensor) -> torch.Tensor:
    if img.dim() == 3:
        img = img.unsqueeze(1)
    return F.max_pool2d(img, kernel_size=3, stride=1, padding=1)

def soft_open(img: torch.Tensor) -> torch.Tensor:
    return soft_dilate(soft_erode(img))

def soft_skel(img: torch.Tensor, num_iter: int = 10) -> torch.Tensor:
    if img.dim() == 3:
        img = img.unsqueeze(1)
    img1 = soft_open(img)
    skel = F.relu(img - img1)
    for _ in range(num_iter):
        img = soft_erode(img)
        img1 = soft_open(img)
        delta = F.relu(img - img1)
        skel = skel + F.relu(delta - skel * delta)
    return skel

class SoftClDiceLoss(nn.Module):
    def __init__(self, num_iter: int = 10, smooth: float = 1.0):
        super().__init__()
        self.num_iter = num_iter
        self.smooth = smooth

    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        if pred.dim() == 3:
            pred = pred.unsqueeze(1)
        if target.dim() == 3:
            target = target.unsqueeze(1)

        skel_pred = soft_skel(torch.sigmoid(pred), self.num_iter)
        skel_target = soft_skel(target, self.num_iter)

        tprec_num = (skel_pred * target).sum(dim=(1, 2, 3)) + self.smooth
        tprec_den = skel_pred.sum(dim=(1, 2, 3)) + self.smooth
        tprec = tprec_num / tprec_den

        tsens_num = (skel_target * torch.sigmoid(pred)).sum(dim=(1, 2, 3)) + self.smooth
        tsens_den = skel_target.sum(dim=(1, 2, 3)) + self.smooth
        tsens = tsens_num / tsens_den

        cl_dice = 2.0 * (tprec * tsens) / (tprec + tsens + 1e-7)
        return (1.0 - cl_dice).mean()

class SoftDiceLoss(nn.Module):
    def __init__(self, smooth: float = 1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        pred = torch.sigmoid(pred)
        if pred.dim() == 3:
            pred = pred.unsqueeze(1)
        if target.dim() == 3:
            target = target.unsqueeze(1)

        intersection = (pred * target).sum(dim=(1, 2, 3))
        cardinality = pred.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
        dice = (2.0 * intersection + self.smooth) / (cardinality + self.smooth)
        return (1.0 - dice).mean()

class RoadExtractionLoss(nn.Module):
    def __init__(
        self,
        total_epochs: int,
        alpha_start: float = 1.0,
        alpha_end: float = 0.1,
        num_iter: int = 10,
        smooth: float = 1.0,
    ):
        super().__init__()
        self.total_epochs = max(total_epochs, 1)
        self.alpha_start = alpha_start
        self.alpha_end = alpha_end
        self.alpha = alpha_start

        self.bce = nn.BCEWithLogitsLoss()
        self.cldice = SoftClDiceLoss(num_iter=num_iter, smooth=smooth)

    def update_alpha(self, epoch: int) -> float:
        decay = (self.alpha_start - self.alpha_end) * epoch / self.total_epochs
        self.alpha = max(self.alpha_end, self.alpha_start - decay)
        return self.alpha

    def forward(
        self,
        logits: torch.Tensor,
        target: torch.Tensor,
        return_components: bool = False,
    ) -> torch.Tensor | Tuple[torch.Tensor, Dict[str, float]]:
        # Force float32 under disabled autocast to avoid BCE/Morphology issues with AMP
        with torch.amp.autocast(device_type=logits.device.type, enabled=False):
            logits_f32 = logits.float()
            target_f32 = target.float()
            bce_loss = self.bce(logits_f32, target_f32)
            cldice_loss = self.cldice(logits_f32, target_f32)

        total_loss = self.alpha * bce_loss + (1.0 - self.alpha) * cldice_loss

        if return_components:
            components = {
                "total_loss": total_loss.item(),
                "bce_loss": bce_loss.item(),
                "cldice_loss": cldice_loss.item(),
                "alpha": self.alpha,
            }
            return total_loss, components

        return total_loss

In [27]:
def _make_divisible(v: float, divisor: int = 8, min_value: int = None) -> int:
    if min_value is None:
        min_value = divisor
    new_v = max(min_value, int(v + divisor / 2) // divisor * divisor)
    if new_v < 0.9 * v:
        new_v += divisor
    return new_v

class StripConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1):
        super().__init__()
        self.conv_h = nn.Conv2d(in_channels, out_channels, kernel_size=(1, 3), stride=(1, stride), padding=(0, 1), bias=False)
        self.bn_h = nn.BatchNorm2d(out_channels)
        self.conv_v = nn.Conv2d(out_channels, out_channels, kernel_size=(3, 1), stride=(stride, 1), padding=(1, 0), bias=False)
        self.bn_v = nn.BatchNorm2d(out_channels)
        self.act = nn.SiLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.act(self.bn_h(self.conv_h(x)))
        x = self.act(self.bn_v(self.conv_v(x)))
        return x

class ChannelShift(nn.Module):
    def __init__(self, shift_pixels: int = 2, shift_fraction: float = 0.25):
        super().__init__()
        self.shift_pixels = shift_pixels
        self.shift_fraction = shift_fraction

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape
        s = self.shift_pixels
        n_shifted = int(C * self.shift_fraction)
        per_dir = n_shifted // 4

        if per_dir == 0 or s == 0:
            return x

        c_up    = x[:, 0 * per_dir : 1 * per_dir]
        c_down  = x[:, 1 * per_dir : 2 * per_dir]
        c_left  = x[:, 2 * per_dir : 3 * per_dir]
        c_right = x[:, 3 * per_dir : 4 * per_dir]
        c_id    = x[:, 4 * per_dir :]

        c_up    = F.pad(c_up[:, :, s:, :],     (0, 0, 0, s))
        c_down  = F.pad(c_down[:, :, :-s, :],  (0, 0, s, 0))
        c_left  = F.pad(c_left[:, :, :, s:],   (0, s, 0, 0))
        c_right = F.pad(c_right[:, :, :, :-s], (s, 0, 0, 0))

        return torch.cat([c_up, c_down, c_left, c_right, c_id], dim=1)

class InvertedResidual(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1, expand_ratio: int = 2):
        super().__init__()
        mid = in_channels * expand_ratio
        self.use_residual = (stride == 1 and in_channels == out_channels)

        layers = []
        if expand_ratio != 1:
            layers.extend([
                nn.Conv2d(in_channels, mid, 1, bias=False),
                nn.BatchNorm2d(mid),
                nn.SiLU(inplace=True),
            ])
        layers.extend([
            nn.Conv2d(mid, mid, 3, stride=stride, padding=1, groups=mid, bias=False),
            nn.BatchNorm2d(mid),
            nn.SiLU(inplace=True),
            nn.Conv2d(mid, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
        ])
        self.conv = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.use_residual:
            return x + self.conv(x)
        return self.conv(x)

class LinearSelfAttention(nn.Module):
    def __init__(self, embed_dim: int, attn_dropout: float = 0.0):
        super().__init__()
        self.embed_dim = embed_dim
        self.qkv = nn.Linear(embed_dim, 2 * embed_dim + 1, bias=True)
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=True)
        self.attn_drop = nn.Dropout(attn_dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        qkv = self.qkv(x)
        q, k, v = qkv.split([self.embed_dim, 1, self.embed_dim], dim=-1)

        context_scores = F.softmax(k, dim=1)
        context_scores = self.attn_drop(context_scores)
        context_vector = (context_scores * v).sum(dim=1, keepdim=True)

        out = F.relu(q) * context_vector
        return self.out_proj(out)

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim: int, ffn_ratio: float = 2.0, dropout: float = 0.0, attn_dropout: float = 0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = LinearSelfAttention(embed_dim, attn_dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        ffn_dim = int(embed_dim * ffn_ratio)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, ffn_dim),
            nn.SiLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(ffn_dim, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x

class MobileViTv2Block(nn.Module):
    def __init__(
        self,
        in_channels: int,
        transformer_dim: int,
        n_transformer_layers: int = 2,
        patch_size: int = 2,
        ffn_ratio: float = 2.0,
        dropout: float = 0.0,
        attn_dropout: float = 0.0,
        shift_pixels: int = 2,
        shift_fraction: float = 0.25,
    ):
        super().__init__()
        self.patch_h = patch_size
        self.patch_w = patch_size
        self.channel_shift = ChannelShift(shift_pixels, shift_fraction)
        self.local_rep = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, 3, padding=1, groups=in_channels, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.SiLU(inplace=True),
            nn.Conv2d(in_channels, transformer_dim, 1, bias=False),
            nn.BatchNorm2d(transformer_dim),
        )
        self.transformers = nn.Sequential(*[
            TransformerBlock(transformer_dim, ffn_ratio, dropout, attn_dropout)
            for _ in range(n_transformer_layers)
        ])
        self.post_norm = nn.LayerNorm(transformer_dim)
        self.proj = nn.Sequential(
            nn.Conv2d(transformer_dim, in_channels, 1, bias=False),
            nn.BatchNorm2d(in_channels),
        )
        self.fusion = nn.Sequential(
            nn.Conv2d(2 * in_channels, in_channels, 1, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.SiLU(inplace=True),
        )

    def _unfold(self, x: torch.Tensor):
        B, C, H, W = x.shape
        ph, pw = self.patch_h, self.patch_w
        n_h, n_w = H // ph, W // pw
        x = x.reshape(B, C, n_h, ph, n_w, pw)
        x = x.permute(0, 3, 5, 1, 2, 4)
        x = x.reshape(B * ph * pw, C, n_h * n_w)
        x = x.permute(0, 2, 1)
        return x, (B, n_h, n_w)

    def _fold(self, x: torch.Tensor, info: tuple, C: int):
        B, n_h, n_w = info
        ph, pw = self.patch_h, self.patch_w
        x = x.permute(0, 2, 1)
        x = x.reshape(B, ph, pw, C, n_h, n_w)
        x = x.permute(0, 3, 4, 1, 5, 2)
        x = x.reshape(B, C, n_h * ph, n_w * pw)
        return x

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C_in, H, W = x.shape
        ph, pw = self.patch_h, self.patch_w
        pad_h = (ph - H % ph) % ph
        pad_w = (pw - W % pw) % pw
        if pad_h > 0 or pad_w > 0:
            x = F.pad(x, (0, pad_w, 0, pad_h))

        identity = x
        x_shifted = self.channel_shift(x)
        local_out = self.local_rep(x_shifted)
        C_t = local_out.shape[1]

        tokens, fold_info = self._unfold(local_out)
        tokens = self.transformers(tokens)
        tokens = self.post_norm(tokens)
        global_out = self._fold(tokens, fold_info, C_t)
        global_out = self.proj(global_out)
        fused = self.fusion(torch.cat([identity, global_out], dim=1))

        if pad_h > 0 or pad_w > 0:
            fused = fused[:, :, :H, :W]
        return fused

class MobileViT_v2(nn.Module):
    def __init__(self, num_classes: int = 1, width_mult: float = 1.0):
        super().__init__()
        def _c(channels: int) -> int:
            return _make_divisible(channels * width_mult)

        self.stem = StripConv(3, _c(32), stride=2)
        self.enc1 = InvertedResidual(_c(32), _c(64), stride=2)
        self.enc2_mvit = MobileViTv2Block(_c(64), transformer_dim=_c(96), n_transformer_layers=2)
        self.enc2_down = InvertedResidual(_c(64), _c(96), stride=2)
        self.enc3_mvit = MobileViTv2Block(_c(96), transformer_dim=_c(144), n_transformer_layers=2)
        self.enc3_down = InvertedResidual(_c(96), _c(128), stride=2)
        self.bottleneck = MobileViTv2Block(_c(128), transformer_dim=_c(192), n_transformer_layers=3)

        self.up3 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.dec3 = StripConv(_c(128) + _c(96), _c(96))
        self.up2 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.dec2 = StripConv(_c(96) + _c(64), _c(64))
        self.up1 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.dec1 = StripConv(_c(64) + _c(32), _c(32))
        self.up0 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.head = nn.Conv2d(_c(32), num_classes, kernel_size=1, bias=True)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        s1 = self.stem(x)
        e1 = self.enc1(s1)
        s2 = self.enc2_mvit(e1)
        e2 = self.enc2_down(s2)
        s3 = self.enc3_mvit(e2)
        e3 = self.enc3_down(s3)
        bn = self.bottleneck(e3)

        d3 = self.dec3(torch.cat([self.up3(bn), s3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), s2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), s1], dim=1))
        out = self.head(self.up0(d1))
        return out

    @property
    def num_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

In [28]:
def train_one_epoch(epoch, model, dataloader, optimizer, scaler, loss_fn, logger, device):
    model.train()
    total_loss = 0.0
    total_bce = 0.0
    total_cldice = 0.0
    current_alpha = loss_fn.update_alpha(epoch)

    pbar = tqdm(dataloader, desc=f"Epoch {epoch} [Train]", leave=False)
    for images, masks in pbar:
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad(set_to_none=True)

        with autocast(device_type=device.type, enabled=(device.type == 'cuda')):
            logits = model(images)
            loss, components = loss_fn(logits, masks, return_components=True)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        total_bce += components['bce_loss']
        total_cldice += components['cldice_loss']
        pbar.set_postfix({"Loss": f"{loss.item():.4f}", "Alpha": f"{current_alpha:.2f}"})

        logger.log_metrics({
            "train/step_loss": loss.item(),
            "train/step_bce": components['bce_loss'],
            "train/step_cldice": components['cldice_loss'],
            "lr": optimizer.param_groups[0]['lr']
        })

    num_batches = len(dataloader)
    return total_loss / num_batches, total_bce / num_batches, total_cldice / num_batches

@torch.no_grad()
def validate(epoch, model, dataloader, loss_fn, device):
    model.eval()
    total_loss = 0.0
    total_bce = 0.0
    total_cldice = 0.0

    pbar = tqdm(dataloader, desc=f"Epoch {epoch} [Val]", leave=False)
    for images, masks in pbar:
        images, masks = images.to(device), masks.to(device)

        with autocast(device_type=device.type, enabled=(device.type == 'cuda')):
            preds = model(images)
            loss, components = loss_fn(preds, masks, return_components=True)

        total_loss += loss.item()
        total_bce += components['bce_loss']
        total_cldice += components['cldice_loss']
        pbar.set_postfix({"Val Loss": f"{loss.item():.4f}"})

    num_batches = len(dataloader)
    return total_loss / num_batches, total_bce / num_batches, total_cldice / num_batches

In [29]:
def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # Path validation
    print("--- Validating Paths ---")
    for name, path in [
        ('Train Images', TRAIN_IMG_DIR),
        ('Train Masks', TRAIN_MASK_DIR),
        ('Val Images', VAL_IMG_DIR),
        ('Val Masks', VAL_MASK_DIR)
    ]:
        if not os.path.exists(path):
            print(f"⚠️ Warning: Path not found for {name}: {path}")
        else:
            print(f"✅ {name} path exists: {path}")

    logger = WandbLogger(
        project="rural-road-extraction",
        run_name=RUN_NAME,
        config={
            "epochs": EPOCHS,
            "batch_size": BATCH_SIZE,
            "lr": LR,
            "width_mult": WIDTH_MULT,
            "num_workers": NUM_WORKERS
        },
        output_dir=OUTPUT_DIR
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # 1. Datasets
    train_dataset = DeepGlobeDataset(
        image_dir=TRAIN_IMG_DIR,
        mask_dir=TRAIN_MASK_DIR,
        transform=get_train_transforms()
    )
    val_dataset = DeepGlobeDataset(
        image_dir=VAL_IMG_DIR,
        mask_dir=VAL_MASK_DIR,
        transform=get_val_transforms()
    )

    if len(train_dataset) == 0:
        raise ValueError(f"No valid image/mask pairs found in training directory: {TRAIN_IMG_DIR}")

    # If the validation set has no masks, dynamically split the training dataset (90/10) for validation.
    if len(val_dataset) == 0:
        print("⚠️ Warning: Validation dataset has no masks. Dynamically splitting training dataset (90/10) for validation...")
        import random
        rng = random.Random(42)
        all_ids = list(train_dataset.ids)
        rng.shuffle(all_ids)

        val_size = max(1, int(len(all_ids) * 0.1))
        train_ids = all_ids[val_size:]
        val_ids = all_ids[:val_size]

        train_dataset.ids = train_ids
        val_dataset.ids = val_ids

        val_dataset.image_dir = TRAIN_IMG_DIR
        val_dataset.mask_dir = TRAIN_MASK_DIR

        print(f"✅ Split dataset into {len(train_dataset)} training and {len(val_dataset)} validation samples.")

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    # 2. Model
    model = MobileViT_v2(num_classes=1, width_mult=WIDTH_MULT).to(device)
    logger.log_config({"num_parameters": model.num_parameters})
    print(f"Model parameters: {model.num_parameters:,}")

    # 3. Loss & Optimizer
    loss_fn = RoadExtractionLoss(total_epochs=EPOCHS, alpha_start=1.0, alpha_end=0.2)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scaler = GradScaler(enabled=(device.type == 'cuda'))

    best_val_cldice = float('inf')

    # 4. Training Loop
    for epoch in range(EPOCHS):
        train_loss, train_bce, train_cldice = train_one_epoch(
            epoch, model, train_loader, optimizer, scaler, loss_fn, logger, device
        )
        val_loss, val_bce, val_cldice = validate(epoch, model, val_loader, loss_fn, device)

        logger.log_metrics({
            "epoch": epoch,
            "train/epoch_loss": train_loss,
            "train/epoch_bce": train_bce,
            "train/epoch_cldice": train_cldice,
            "val/epoch_loss": val_loss,
            "val/epoch_bce": val_bce,
            "val/epoch_cldice": val_cldice,
            "alpha": loss_fn.alpha
        }, step=epoch)

        print(f"Epoch [{epoch}/{EPOCHS-1}] - "
              f"Train Loss: {train_loss:.4f} (clDice: {train_cldice:.4f}) | "
              f"Val Loss: {val_loss:.4f} (clDice: {val_cldice:.4f})")

        if val_cldice < best_val_cldice:
            best_val_cldice = val_cldice
            save_path = os.path.join(OUTPUT_DIR, "best_model.pth")
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_cldice': best_val_cldice,
            }, save_path)
            print(f"--> Saved new best model to {save_path} (Val clDice: {best_val_cldice:.4f})")
            logger.log_artifact(save_path, artifact_type="model", name="best_model")

    logger.log_summary({"best_val_cldice": best_val_cldice})
    logger.finish()

if __name__ == '__main__':
    main()

--- Validating Paths ---
✅ Train Images path exists: /kaggle/working/dataset/train
✅ Train Masks path exists: /kaggle/working/dataset/train
✅ Val Images path exists: /kaggle/working/dataset/valid
✅ Val Masks path exists: /kaggle/working/dataset/valid
[WandbLogger] W&B not available. Logging locally to /kaggle/working/model/wandb_local.json
Using device: cuda
⚠️ Warning: Validation dataset has no masks. Dynamically splitting training dataset (90/10) for validation...
✅ Split dataset into 5604 training and 622 validation samples.
Model parameters: 1,586,568


Epoch [0/49] - Train Loss: 0.1855 (clDice: 0.9036) | Val Loss: 0.1658 (clDice: 0.9207)
--> Saved new best model to /kaggle/working/model/best_model.pth (Val clDice: 0.9207)
[WandbLogger] Artifact (local only): /kaggle/working/model/best_model.pth


Epoch [1/49] - Train Loss: 0.1354 (clDice: 0.8181) | Val Loss: 0.1778 (clDice: 0.8777)
--> Saved new best model to /kaggle/working/model/best_model.pth (Val clDice: 0.8777)
[WandbLogger] Artifact (local only): /kaggle/working/model/best_model.pth


Epoch [2/49] - Train Loss: 0.1259 (clDice: 0.7263) | Val Loss: 0.1894 (clDice: 0.8793)


Epoch [3/49] - Train Loss: 0.1240 (clDice: 0.6348) | Val Loss: 0.2085 (clDice: 0.8487)
--> Saved new best model to /kaggle/working/model/best_model.pth (Val clDice: 0.8487)
[WandbLogger] Artifact (local only): /kaggle/working/model/best_model.pth


...

Epoch [45/49] - Train Loss: 0.3434 (clDice: 0.4121) | Val Loss: 0.6883 (clDice: 0.8576)


Epoch [46/49] - Train Loss: 0.3189 (clDice: 0.3731) | Val Loss: 0.7074 (clDice: 0.8685)


Epoch [47/49] - Train Loss: 0.3325 (clDice: 0.3852) | Val Loss: 0.7286 (clDice: 0.8810)


Epoch [48/49] - Train Loss: 0.3099 (clDice: 0.3512) | Val Loss: 0.6992 (clDice: 0.8227)


Epoch [49/49] - Train Loss: 0.3083 (clDice: 0.3428) | Val Loss: 0.7018 (clDice: 0.8162)
[WandbLogger] Run finished. Local log: /kaggle/working/model/wandb_local.json
